# MLE — Train Arcade Game RL Agent on Colab GPU

Runs MAME headless on Colab + trains PPO/DQN with GPU acceleration.

**Requirements**: Upload your ROM zip (e.g. `galaga.zip`) and the MLE code.

In [ ]:
# 1. Install MAME and dependencies
!apt-get update -qq && apt-get install -y -qq mame > /dev/null 2>&1
!pip install -q gymnasium stable-baselines3 pytesseract pillow
!which mame && mame -version

In [ ]:
# 2. Clone MLE repo
!git clone https://github.com/lilwg/mle.git
%cd mle

In [ ]:
# 3. Upload ROM file
# Option A: Upload from local machine
from google.colab import files
uploaded = files.upload()  # Upload e.g. galaga.zip

# Move ROM to roms directory
import os
os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
    print(f'ROM: roms/{fname}')

In [ ]:
# 4. Fix MAME binary path for Linux
import subprocess
mame_path = subprocess.run(['which', 'mame'], capture_output=True, text=True).stdout.strip()
print(f'MAME at: {mame_path}')

# Patch the code to use Linux MAME path
for pyfile in ['mle_general.py', 'mle/console.py']:
    with open(pyfile) as f:
        code = f.read()
    code = code.replace('/opt/homebrew/bin/mame', mame_path)
    with open(pyfile, 'w') as f:
        f.write(code)
    print(f'Patched {pyfile}')

# Also fix ROMS_PATH
for pyfile in ['mle_general.py', 'validate_addrs.py', 'find_score_ram.py', 'find_ram_addrs.py']:
    if os.path.exists(pyfile):
        with open(pyfile) as f:
            code = f.read()
        code = code.replace('/Users/pat/mame/roms', os.path.abspath('roms'))
        with open(pyfile, 'w') as f:
            f.write(code)
        print(f'Patched {pyfile}')

In [ ]:
# 5. Quick test: verify MAME + MLE works
!python3 -c "
import sys; sys.path.insert(0, '.')
from mle import MameEnv
env = MameEnv('roms', 'galaga', {'_dummy': 0}, render=False, sound=False, throttle=False)
env.wait(100)
d = env.step()
print(f'MAME works! Got {len(d)} RAM values')
env.close()
" 2>&1 | grep -v LUA

In [ ]:
# 6. Train! GPU-accelerated PPO on Galaga
# Change 'galaga' to your game, adjust timesteps as needed
GAME = 'galaga'
TIMESTEPS = 5_000_000  # 5M steps — ~3-6 hours on Colab GPU
MODEL = 'ppo'

!python3 mle_general.py {GAME} \
    --model {MODEL} \
    --timesteps {TIMESTEPS} \
    --save {GAME}_{MODEL}_{TIMESTEPS//1000}k

In [ ]:
# 7. Download trained model
from google.colab import files
import glob
for f in glob.glob('*.zip'):
    print(f'Downloading {f}...')
    files.download(f)